# Istanbul Metro Service Planner — analysis

**Question:** on which Istanbul rail line, and in which time band, is the load per trip uneven — where could service be added or reduced?

This notebook rebuilds every number in the README and the panel from the rail aggregate (`data/raw/rail_hourly/*.parquet`, made by `scripts/build_data.py`). Each step calls a function in `metro/`; the cells only show and compare. Where a number is also in `app/data.json`, the cell checks that the two agree.

Runtime: about three minutes (the special-day backtests take most of it).

In [1]:
import json, sys
from pathlib import Path
import duckdb
import pandas as pd

ROOT = Path.cwd()
assert (ROOT / "metro").is_dir(), "run this notebook from the repository root"
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "app"))

from metro.prep import ANALYSIS_END, MIN_DAY_BOARDINGS, STRAY_LINES, analysis_hourly, complete_days, load_hourly
from metro.calendar import load_calendar
from metro.features import FEATURE_COLS, station_features
from metro.groups import OUTLIER, discover_groups, fit_rule, rule_agreement, rule_text
from metro.forecast import CLOSED_THRESHOLD, special_factors, typical_profile, wape
from metro import backtest as bt
from metro.app_data import EXTRA, MAIN, choose_level, recommendation_rows
import model

pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
PARQUET = ROOT / "data" / "raw" / "rail_hourly"
APP = json.loads((ROOT / "app" / "data.json").read_text())
BT = APP["meta"]["backtest"]
pct = lambda x: f"{x * 100:.1f}%"

## 1 · Question and data

Two IBB Open Data Portal datasets: hourly public-transport boardings (monthly CSVs, rail rows only) and monthly trips per rail line. The unit of analysis is a line–station pair (a station's gates for one line). Demand is `number_of_passage` (boardings at gates).

In [2]:
full = load_hourly(PARQUET)
full = full[~full["line"].isin(STRAY_LINES)]
cal = load_calendar()
h = analysis_hourly(full)          # complete days up to ANALYSIS_END only
days_all = complete_days(full)
print(f"aggregate rows: {len(full):,}; dates {full.date.min():%Y-%m-%d} .. {full.date.max():%Y-%m-%d}")
print(f"analysis period: {h.date.min():%Y-%m-%d} .. {h.date.max():%Y-%m-%d}, {h.date.nunique()} days, "
      f"{h.groupby(['line', 'station']).ngroups} line-station units, {h.line.nunique()} lines")
assert h.date.min().strftime("%Y-%m-%d") == APP["meta"]["data_start"] and ANALYSIS_END == APP["meta"]["data_end"]

aggregate rows: 3,874,987; dates 2023-01-01 .. 2024-10-18
analysis period: 2023-01-01 .. 2024-07-31, 578 days, 383 line-station units, 23 lines


## 2 · Data prep

Each monthly CSV (0.3–1.8 GB) is reduced with one DuckDB query to rail boardings per line, station, date and hour, then deleted. The query:

In [3]:
print((ROOT / "sql" / "rail_hourly.sql").read_text())

-- One month of IBB hourly public transport data -> rail boardings per
-- line, station, date and hour.
-- {csv_path} is filled in by metro/prep.py; every column is read as text and cast here.
SELECT
    CAST(transition_date AS DATE)                                  AS date,
    CAST(transition_hour AS INTEGER)                               AS hour,
    trim(line_name)                                                AS line,
    coalesce(nullif(upper(trim(station_poi_desc_cd)), ''), 'UNKNOWN') AS station,
    CAST(sum(CAST(number_of_passage AS BIGINT)) AS BIGINT)         AS boardings,
    CAST(sum(CAST(number_of_passenger AS BIGINT)) AS BIGINT)       AS passengers,
    CAST(sum(CASE WHEN product_kind = 'TAM'
                  THEN CAST(number_of_passage AS BIGINT) ELSE 0 END) AS BIGINT) AS b_full,
    CAST(sum(CASE WHEN product_kind LIKE 'INDIRIMLI%'
                  THEN CAST(number_of_passage AS BIGINT) ELSE 0 END) AS BIGINT) AS b_discount,
    CAST(sum(CASE WHEN product_kind = 'UCRE

Monthly totals straight from the parquet files with DuckDB. From August 2024 most days in the source files are nearly empty, so a *complete day* is one with at least 100,000 rail boardings; the analysis stops at July 2024.

In [4]:
monthly = duckdb.sql(f'''
    WITH d AS (SELECT date, sum(boardings) AS b FROM read_parquet('{PARQUET}/*.parquet') GROUP BY date)
    SELECT strftime(date, '%Y-%m') AS month, count(*) AS days_in_file,
           count(*) FILTER (WHERE b >= {MIN_DAY_BOARDINGS}) AS complete_days,
           sum(b) AS boardings, max(b) FILTER (WHERE b < {MIN_DAY_BOARDINGS}) AS largest_thin_day
    FROM d GROUP BY 1 ORDER BY 1''').df()
monthly

,month,days_in_file,complete_days,boardings,largest_thin_day
0,2023-01,31,31,89291579.0,NaN
1,2023-02,28,28,71991235.0,NaN
2,2023-03,31,31,86095180.0,NaN
3,2023-04,30,30,84274076.0,NaN
4,2023-05,31,31,91943898.0,NaN
5,2023-06,30,30,84390258.0,NaN
6,2023-07,31,31,80772845.0,NaN
7,2023-08,31,31,81478496.0,NaN
8,2023-09,30,30,85801583.0,NaN
9,2023-10,31,31,102664360.0,NaN


In [5]:
after = days_all[days_all > pd.Timestamp(ANALYSIS_END)]
print(f"complete days after July 2024: {len(after)} ->", {m: int(n) for m, n in pd.Series(after.strftime('%Y-%m')).value_counts().sort_index().items()})
print(f"complete days Jan 2023 - Jul 2024: {(days_all <= pd.Timestamp(ANALYSIS_END)).sum()}")

complete days after July 2024: 19 -> {'2024-08': 6, '2024-09': 8, '2024-10': 5}
complete days Jan 2023 - Jul 2024: 578


## 3 · Measured indicator: boardings per trip

Boardings per trip = a month's boardings on a line ÷ the trips that line ran that month. It needs no assumption, but it is **not occupancy**: exits, trip length and train size are not in the data. Compare a line with itself over time, not with other lines.

In [6]:
trips = pd.read_csv(ROOT / "data" / "trips_monthly.csv", dtype={"month": str})
mb = h.assign(month=h.date.dt.strftime("%Y-%m")).groupby(["line", "month"])["boardings"].sum().reset_index()
bpt = mb.merge(trips, on=["line", "month"]).assign(per_trip=lambda d: d.boardings / d.trips)
july = bpt[bpt.month.isin(["2023-07", "2024-07"])].pivot(index="line", columns="month", values=["boardings", "trips", "per_trip"])
cmp = pd.DataFrame({
    "per_trip_jul23": july[("per_trip", "2023-07")].round(0), "per_trip_jul24": july[("per_trip", "2024-07")].round(0),
    "per_trip_change": (july[("per_trip", "2024-07")] / july[("per_trip", "2023-07")] - 1).round(3),
    "boardings_change": (july[("boardings", "2024-07")] / july[("boardings", "2023-07")] - 1).round(3),
    "trips_change": (july[("trips", "2024-07")] / july[("trips", "2023-07")] - 1).round(3)})
cmp.sort_values("per_trip_change")

,per_trip_jul23,per_trip_jul24,per_trip_change,boardings_change,trips_change
line,,,,,
TF1,11.0,9.0,-0.194,-0.353,-0.197
TF2,25.0,22.0,-0.095,-0.022,0.080
T1,513.0,474.0,-0.076,-0.080,-0.004
F1,47.0,45.0,-0.051,-0.046,0.006
M2,688.0,654.0,-0.050,-0.040,0.011
M6,52.0,51.0,-0.028,-0.009,0.019
F4,12.0,12.0,0.014,0.016,0.002
T4,424.0,432.0,0.017,-0.022,-0.039
M4,696.0,732.0,0.052,0.045,-0.007


In [7]:
# the same figures from app/data.json, which the panel and README use
for line, x in APP["lines"].items():
    m = {r["month"]: r for r in x["monthly"]}
    if x["has_trips"] and line in cmp.index:
        assert round(m["2024-07"]["boardings"] / m["2024-07"]["trips"]) == cmp.at[line, "per_trip_jul24"], line
print("data.json agrees with the recomputed July figures for", int(cmp.per_trip_jul24.notna().sum()), "lines")

data.json agrees with the recomputed July figures for 17 lines


In [8]:
bpt.pivot(index="month", columns="line", values="per_trip").round(0)

line,F1,F4,M1,M2,M3,M4,M5,M6,M7,M8,M9,T1,T3,T4,T5,TF1,TF2
month,,,,,,,,,,,,,,,,,
2023-01,41.0,11.0,503.0,734.0,214.0,771.0,693.0,66.0,343.0,177.0,81.0,486.0,43.0,436.0,93.0,6.0,11.0
2023-02,32.0,10.0,457.0,641.0,203.0,663.0,605.0,57.0,309.0,162.0,76.0,455.0,38.0,401.0,79.0,11.0,21.0
2023-03,36.0,12.0,482.0,685.0,220.0,718.0,667.0,73.0,336.0,190.0,79.0,475.0,43.0,435.0,87.0,11.0,20.0
2023-04,40.0,14.0,505.0,689.0,255.0,705.0,672.0,67.0,341.0,193.0,75.0,474.0,43.0,439.0,107.0,13.0,21.0
2023-05,43.0,14.0,513.0,716.0,287.0,751.0,706.0,71.0,353.0,197.0,74.0,527.0,47.0,456.0,112.0,5.0,57.0
2023-06,46.0,13.0,499.0,696.0,252.0,703.0,652.0,60.0,335.0,187.0,76.0,515.0,45.0,440.0,118.0,13.0,24.0
2023-07,47.0,12.0,513.0,688.0,274.0,696.0,622.0,52.0,335.0,193.0,71.0,513.0,39.0,424.0,101.0,11.0,25.0
2023-08,50.0,11.0,521.0,668.0,278.0,745.0,623.0,49.0,339.0,190.0,71.0,526.0,39.0,420.0,74.0,10.0,24.0
2023-09,49.0,11.0,556.0,706.0,289.0,758.0,665.0,60.0,364.0,222.0,74.0,505.0,44.0,444.0,216.0,10.0,24.0


## 4 · Station features and groups

Seven features per line-station (non-special days only): weekend/weekday ratio, the weekday shares of the morning peak (07–09), midday (10–15), evening peak (16–19) and late hours (20–05), and the discounted-fare and transfer shares. K-means on standardized features, k from 4–6 by silhouette; a cluster smaller than 5 stations is marked as an outlier. A depth-3 decision tree turns the clusters into a short rule.

In [9]:
features = station_features(h, cal)
groups = discover_groups(features)
names = {i: g["name"] for i, g in enumerate(APP["groups"]) if g["name"] != "Event-driven (unpredictable)"}
names[OUTLIER] = "Event-driven (unpredictable)"
d = features.loc[groups.index].assign(group=groups.map(names))
summary = d.groupby("group")[FEATURE_COLS].mean().round(2).assign(stations=d.groupby("group").size())
summary

,weekend_ratio,am_share,midday_share,pm_share,late_share,discount_share,transfer_share,stations
group,,,,,,,,
All-day & weekend,0.85,0.14,0.36,0.35,0.13,0.39,0.08,73
"Evening departure, quiet weekend",0.49,0.13,0.25,0.52,0.10,0.51,0.06,51
Event-driven (unpredictable),0.01,0.00,0.01,0.03,0.96,0.53,0.02,1
Morning departure,0.70,0.33,0.30,0.25,0.08,0.51,0.05,147
Transfer hub,0.74,0.19,0.26,0.39,0.14,0.55,0.25,49


In [10]:
assert {g["name"]: g["n_stations"] for g in APP["groups"]} == summary["stations"].to_dict()
for g in APP["groups"]:
    print(f"{g['name']:34s} {g['n_stations']:4d}  e.g. {', '.join(g['examples'])}")
print(f"\n{len(features) - len(groups)} units have under 12 months of data and are not grouped (demand view only).")

Morning departure                   147  e.g. MARMARAY USKUDAR-2, M5 CARSI, M5 UMRANIYE, M1 BAGCILAR MEYDAN, MARMARAY ZEYTINBURNU
All-day & weekend                    73  e.g. M1 ATAKOY, T1 SIRKECI, T1 SULTANAHMET, MARMARAY PENDIK, MARMARAY BAKIRKOY-2
Evening departure, quiet weekend     51  e.g. M2 OSMANBEY KUZEY, M2 LEVENT 2 KUZEY, M2 ITU GUNEY, MARMARAY FENERYOLU, M4 YENISAHRA
Transfer hub                         49  e.g. M1 YENIKAPI, M5 USKUDAR 1, M2 GAYRETTEPE, M4 AYRILIKCESME, T1 ZEYTINBURNU 2
Event-driven (unpredictable)          1  e.g. 

62 units have under 12 months of data and are not grouped (demand view only).


In [11]:
tree = fit_rule(features, groups)
agree = rule_agreement(tree, features, groups)
print(rule_text(tree))
print(f"rule matches the clusters for {agree:.1%} of stations")
assert round(agree, 4) == APP["meta"]["rule_agreement"]
print("\nIn words:", APP["meta"]["rule"])

|--- am_share <= 0.23
|   |--- pm_share <= 0.40
|   |   |--- discount_share <= 0.44
|   |   |   |--- class: 1
|   |   |--- discount_share >  0.44
|   |   |   |--- class: 1
|   |--- pm_share >  0.40
|   |   |--- transfer_share <= 0.14
|   |   |   |--- class: 2
|   |   |--- transfer_share >  0.14
|   |   |   |--- class: 3
|--- am_share >  0.23
|   |--- transfer_share <= 0.18
|   |   |--- pm_share <= 0.41
|   |   |   |--- class: 0
|   |   |--- pm_share >  0.41
|   |   |   |--- class: 2
|   |--- transfer_share >  0.18
|   |   |--- late_share <= 0.07
|   |   |   |--- class: 0
|   |   |--- late_share >  0.07
|   |   |   |--- class: 3

rule matches the clusters for 83.2% of stations

In words: If more than 23% of a station's weekday boardings are in the morning peak (07-09) and fewer than 18% are transfers, it is a Morning departure station, unless over 41% fall in the evening peak (16-19), which makes it Evening departure. If the morning share is 23% or less: up to 40% in the evening peak me

In [12]:
# the outlier: one station whose boardings come almost entirely after 20:00 (event traffic)
features.loc[groups[groups == OUTLIER].index, FEATURE_COLS + ["months_active"]]

,,weekend_ratio,am_share,midday_share,pm_share,late_share,discount_share,transfer_share,months_active
line,station,,,,,,,,
M9,OLIMPIYAT DOGU,0.013839,0.000753,0.010668,0.032631,0.955949,0.532744,0.015314,19


## 5 · Forecast method

`boardings(station, day, hour) = typical profile × special-day factor`

- **Typical profile:** mean boardings per day type (weekday / Saturday / Sunday) and hour over the 8 normal weeks before the cut. Closed-station days (a normal day below 20% of that station's median for the day type, median from earlier days only) are left out.
- **Special-day factor:** for each earlier public holiday, religious holiday or school-break day, actual ÷ the station's same-day-type average of the 28 days before it; the factor is the median ratio. It can be learned per station, per group (pooled) or not at all (`none`).
- Everything is built only from data before the cut, so a backtest never sees the days it scores.

Example: the M2 Taksim south entrance (`TAKSIM GUNEY`; station names in the data are often entrances) on the eve of the summer scenario.

In [13]:
cut = pd.Timestamp("2024-08-01")
key = ("M2", "TAKSIM GUNEY")
hk = h[(h.line == key[0]) & (h.station == key[1])]
assert len(hk), key
prof = typical_profile(hk, cal, as_of=cut, closed_threshold=CLOSED_THRESHOLD)
print(prof.pivot(index="hour", columns="day_type", values="boardings").loc[6:23].round(0).T.to_string())
fac = special_factors(hk, cal, groups, before=cut, level="station", closed_threshold=CLOSED_THRESHOLD)
fac

hour         6      7       8       9       10      11      12      13      14      15      16      17      18      19      20      21      22      23
day_type                                                                                                                                              
saturday  346.0  469.0   519.0   595.0   806.0   981.0  1092.0  1227.0  1372.0  1827.0  1941.0  2121.0  2177.0  2019.0  1861.0  1909.0  2042.0  2494.0
sunday    388.0  426.0   411.0   497.0   566.0   708.0   766.0   831.0   953.0  1342.0  1365.0  1517.0  1587.0  1558.0  1551.0  1564.0  1780.0  1870.0
weekday   384.0  829.0  1031.0  1094.0  1288.0  1491.0  1576.0  1509.0  1855.0  2524.0  2940.0  3626.0  3958.0  2496.0  1995.0  2033.0  2401.0  2260.0


,line,station,special,factor
0,F1,KABATAS,none,1.0
1,F1,KABATAS,public_holiday,1.0
2,F1,KABATAS,religious_holiday,1.0
3,F1,KABATAS,school_break,1.0
4,F1,TAKSIM,none,1.0
...,...,...,...,...
1275,TUNEL,BEYOGLU,school_break,1.0
1276,TUNEL,KARAKOY,none,1.0
1277,TUNEL,KARAKOY,public_holiday,1.0
1278,TUNEL,KARAKOY,religious_holiday,1.0


## 6 · Backtests

Error measures: **WAPE** = Σ|actual − forecast| ÷ Σ actual (boardings-weighted; the headline) and MAPE = mean of |actual − forecast| ÷ actual per station-day (unweighted; small stations and closed-station days inflate it). Station-day errors include hours 0–5; line-band errors cover only the card's bands (06–23).

**Main backtest** April–May 2024 (school term) and **extra** June–July 2024 (start of summer), for the three special-day factor levels.

In [14]:
rows = []
for name, (s, e) in {"main Apr-May 2024": MAIN, "extra Jun-Jul 2024": EXTRA}.items():
    for lv in bt.LEVELS:
        r = bt.backtest_recent(h, cal, groups, start=s, end=e, level=lv, closed_threshold=CLOSED_THRESHOLD)
        rows.append({"test": name, "level": lv, **{k: round(v, 4) for k, v in r.items()}})
recent = pd.DataFrame(rows)
recent

,test,level,station_day_mape,line_band_mape,station_day_wape,line_band_wape
0,main Apr-May 2024,none,0.7773,0.3621,0.1257,0.1191
1,main Apr-May 2024,station,0.7956,0.3262,0.1067,0.1099
2,main Apr-May 2024,group,0.8187,0.3394,0.1156,0.1125
3,extra Jun-Jul 2024,none,0.3193,0.4503,0.2312,0.2399
4,extra Jun-Jul 2024,station,0.1986,0.3527,0.1522,0.1694
5,extra Jun-Jul 2024,group,0.2261,0.3946,0.1575,0.1693


In [15]:
for r in recent.itertuples():
    block = BT["main"] if r.test.startswith("main") else BT["extra"]
    assert block[r.level]["station_day_wape"] == r.station_day_wape and block[r.level]["line_band_wape"] == r.line_band_wape
print("matches app/data.json; headline forecast error (main, station level, station-day WAPE):", pct(BT["station_day_wape"]))

matches app/data.json; headline forecast error (main, station level, station-day WAPE): 10.7%


**Special days of 2024**, each block forecast from data before its first day. This is where the three factor levels differ, and where groups were expected to help.

In [16]:
special = bt.backtest_special(h, cal, groups, closed_threshold=CLOSED_THRESHOLD)
sp = special.pivot(index="special", columns="level", values="station_day_wape").round(4)
sp["days"] = special.groupby("special").n_days.first()
sp

level,group,none,station,days
special,,,,
public_holiday,0.3672,0.3718,0.3321,5
religious_holiday,0.3189,0.4997,0.2356,9
school_break,0.1084,0.2113,0.0962,40


In [17]:
level = choose_level(special, {lv: bt.backtest_recent(h, cal, groups, *MAIN, level=lv, closed_threshold=CLOSED_THRESHOLD) for lv in bt.LEVELS})
assert level == BT["level_used"]
station_wins = all(sp.loc[s, "station"] < sp.loc[s, "group"] for s in sp.index) and \
               all(recent[(recent.test == t) & (recent.level == "station")].station_day_wape.item()
                   < recent[(recent.test == t) & (recent.level == "group")].station_day_wape.item() for t in recent.test.unique())
print(f"level used: {level}")
print("Did groups improve the forecast?", "No: station-level factors beat group-level factors in every test." if station_wins
      else "Mixed: see the tables above.")

level used: station
Did groups improve the forecast? No: station-level factors beat group-level factors in every test.


**The complete days after the cut.** Only 19 days in August–October 2024 are complete in the source files. Each is forecast from data up to July 2024 and compared at network-day level (all lines summed).

In [18]:
cd = {lv: bt.compare_complete_days(full, cal, groups, level=lv, closed_threshold=CLOSED_THRESHOLD) for lv in bt.LEVELS}
c = cd[level].assign(month=lambda d: d.date.dt.strftime("%Y-%m"))
by_month = c.groupby("month").apply(lambda g: pd.Series({"days": len(g), "actual": g.actual.sum(), "forecast": round(g.forecast.sum()),
                                                         "forecast_vs_actual": round(g.forecast.sum() / g.actual.sum() - 1, 4)}))
print({lv: pct(wape(x.actual, x.forecast)) for lv, x in cd.items()}, "WAPE over", len(c), "days")
by_month

{'none': '15.3%', 'station': '10.6%', 'group': '10.7%'} WAPE over 19 days


,days,actual,forecast,forecast_vs_actual
month,,,,
2024-08,6.0,13576370.0,16108030.0,0.1865
2024-09,8.0,18877417.0,21307009.0,0.1287
2024-10,5.0,16104168.0,16282461.0,0.0111


In [19]:
assert len(c) == len(BT["complete_days"]) and round(wape(c.actual, c.forecast), 4) == BT["complete_days_wape"]
aug = by_month.loc["2024-08"]
print(f"Summer scenario check: on the {int(aug.days)} complete August days the forecast was {pct(aug.forecast_vs_actual)} above the actual.")

Summer scenario check: on the 6 complete August days the forecast was 18.6% above the actual.


## 7 · Recommendation card at base assumptions

Trips are published per month only. The card spreads a line's monthly trips over the day under an **assumption** (base: the same number of trips every operating hour, every day), divides the August forecast boardings in each band by those trips, and compares each band with the line's own trip-weighted average: at least 1.25× → *load per trip high*, at most 0.6× → *load per trip low*, otherwise *near average*. Hours 0–5 are left out.

In [20]:
rec = recommendation_rows(APP)
assert rec.equals(pd.read_csv(ROOT / "data" / "recommendations.csv")[rec.columns].astype(rec.dtypes)) or \
       (rec.round(1).astype(str).values == pd.read_csv(ROOT / "data" / "recommendations.csv").round(1).astype(str).values).all()
show = rec.assign(cell=lambda d: d.label.map({"add": "HIGH", "reduce": "low", "keep": "·"}) + d.robust.map({True: "", False: "?"}))
show[show.line == "M2"].pivot(index="day_type", columns="band", values="cell")[list(model.BANDS)]

band,early,am_peak,midday,pm_peak,evening,late
day_type,,,,,,
saturday,low,·,·?,HIGH,·?,·
sunday,low,low,·,·?,·,·
weekday,low,HIGH?,·?,HIGH?,·,·?


## 8 · Sensitivity: which readings survive the assumptions

Two sliders change the assumption: weekday peak hours get 1.0×–2.0× the trips of other hours, and weekends 0.6×–1.0× of weekdays. A reading is **robust** when it is the same at all 25 combinations.

In [21]:
print(f"robust cells: {rec.robust.sum()} of {len(rec)} ({rec.robust.mean():.0%})")
print(rec[rec.label != "keep"].groupby(["label", "robust"]).size().unstack().rename(index={"add": "high", "reduce": "low"}))
peaks = rec[(rec.day_type == "weekday") & rec.band.isin(["am_peak", "pm_peak"]) & (rec.label == "add")]
print(f"\nweekday peak bands read 'high' at base: {len(peaks)}, of which robust: {int(peaks.robust.sum())}")
rec.groupby("line").robust.sum().rename("robust of 18").to_frame().T

robust cells: 181 of 306 (59%)
robust  False  True 
label               
high       27     35
low        19     78

weekday peak bands read 'high' at base: 26, of which robust: 0


line,F1,F4,M1,M2,M3,M4,M5,M6,M7,M8,M9,T1,T3,T4,T5,TF1,TF2
robust of 18,10,11,8,11,7,8,11,12,7,9,10,12,14,9,12,15,15


## 9 · What we can't measure

- Trips are published per month only; how they are spread over hours and weekends is an assumption (the sliders).
- Boardings are counted at gates. Exits, trip length and train size are not in the data, so boardings per trip is not occupancy, and lines are not compared with each other.
- Card readings are relative to each line's own average, so every line shows some high and some low bands; the card says where service looks uneven, not whether a line needs more trips overall.
- The source files are incomplete from August 2024 (19 complete days in August–October). The analysis stops at July 2024; August is a summer scenario.
- The school-break factor is learned from the first weeks of summer, which were busier than August; this is why the August forecast runs high.
- One public-holiday factor: Sunday holidays run above a normal Sunday and weekday holidays below a weekday, so the factor ends up near 1.0. Only 5 public-holiday days fall in the 2024 test.
- Station-day errors include hours 0–5 (under 1% of boardings); line-band errors do not.
- M9 Olimpiyat Doğu is event-driven and gets no forecast; it is shown as demand only.